In [1]:
import os
import warnings
import logging
%pip install -q sktime
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'
os.environ['ABSL_CPP_MIN_LOG_LEVEL'] = '3'

warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('absl').setLevel(logging.ERROR)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.validation import check_is_fitted
import sys

sys.path.append('/kaggle/input/datasets/keithmarange/mini-rocket-methods/')
sys.path.append('/kaggle/input/cmi-competition-code')
import utils

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(3)

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import ParameterSampler, RandomizedSearchCV
import importlib

from scipy.stats import randint, uniform, loguniform

tf.keras.backend.clear_session()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 8.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


E0000 00:00:1775949948.953125      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775949949.040877      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775949949.762720      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775949949.762775      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775949949.762778      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775949949.762781      16 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
data_folder = utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
model_run_name = 'mini_rocket_v1'

feat_columns = ['sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation', 'behavior', 'phase', 'gesture']
acc_columns  = ['acc_x', 'acc_y', 'acc_z']
rot_columns  = ['rot_w', 'rot_x', 'rot_y', 'rot_z']
thm_columns  = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']

non_device_cols = [
    'sequence_type', 'sequence_id', 'sequence_counter', 'subject', 'orientation',
    'behavior', 'phase', 'gesture', 'adult_child', 'age', 'sex', 'handedness',
    'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm'
]

sampling_rate = 100
pipe_name = 'temporal_extractor'

n_splits = 3
tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]

orientation_cols_dict = {
    'Lie on Back': ['Lie on Back'],
}

model_target_list = ['gesture_action']

do_report    = False
save_model   = False
random_search = False
train_size = 0.2

In [4]:
train_df = raw_train_df.set_index('row_id')
target_only_df = train_df[train_df['sequence_type'] == 'Target'].copy()

target_only_df['gesture_position'] = target_only_df['gesture'].str.split(' - ').str[0]
target_only_df['gesture_action']   = target_only_df['gesture'].str.split(' - ').str[-1]

train_sample_df, test_sample_df = utils.sample_balanced_split(
    target_only_df,
    train_pct=train_size,
    test_pct=0.2
)

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%


In [5]:
if random_search:
    param_grid = {
        # RawSequenceExtractor (same as before)
        f'{pipe_name}__acc_cols':      [acc_columns],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [tof_columns],
        f'{pipe_name}__acc_mode':      ['displacement'],
        f'{pipe_name}__rotation_mode': ['delta_euler'],
        f'{pipe_name}__thm_mode':      ['centered'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        'sequence_builder__maxlen':         randint(10, 120),  # Integer range
        'sequence_builder__padding_value':  [-999.0],

        # ============ MiniROCKET Parameters ============
        'classifier__estimator__num_kernels': [1000, 2000, 5000, 10000],
        'classifier__estimator__random_state': [42],
    }
else:
    param_grid = {
        # RawSequenceExtractor
        f'{pipe_name}__acc_cols':      [None],
        f'{pipe_name}__rot_cols':      [rot_columns],
        f'{pipe_name}__thm_cols':      [thm_columns],
        f'{pipe_name}__tof_cols':      [None],
        f'{pipe_name}__acc_mode':      ['time'],
        f'{pipe_name}__rotation_mode': ['euler', 'delta_euler', 'quaternion'],
        f'{pipe_name}__thm_mode':      ['raw'],
        f'{pipe_name}__tof_mode':      ['baseline'],

        'sequence_builder__maxlen':         [110],
        'sequence_builder__padding_value':  [-999.0],

        # ============ MiniROCKET Fixed Parameters ============
        'classifier__estimator__num_kernels': [1000],
        'classifier__estimator__random_state': [42],
    }

In [6]:
importlib.reload(utils)

raw_extractor = utils.RawSequenceExtractor(
    acc_cols=acc_columns,
)

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), make_column_selector(pattern='acc|rot|thm|tof')),
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

sequence_builder = utils.SequencePadder(maxlen=110, padding_value=-999.0)
rocket_clf = utils.MiniRocketClassifier(num_kernels=1000, random_state=42)
classifier = utils.ManyToOneWrapperRNN(estimator=rocket_clf, target='gesture_action')

pipeline = Pipeline([
    (pipe_name, raw_extractor),
    ('preprocessor', preprocessor),
    ('sequence_builder', sequence_builder),
    ('classifier', classifier),
])

cv = GroupKFold(n_splits=n_splits)

In [7]:
for model_target in model_target_list:

    cv_results_list = []
    for col in orientation_cols_dict:
        if random_search:
            search_obj = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=param_grid,
                n_iter=2,
                cv=cv,
                random_state=42,
                n_jobs=1,
                verbose=1,
                return_train_score=True
            )
        else:
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=cv,
                verbose=1,
                n_jobs=1,
                return_train_score=True
            )

        sliced_data_df = train_sample_df[train_sample_df['orientation'].isin(orientation_cols_dict[col])]
        y = sliced_data_df[['sequence_id', model_target]]
        groups = sliced_data_df['sequence_id']

        search_obj.fit(sliced_data_df, y, groups=groups)

        if save_model:
            model = search_obj.best_estimator_
            path_to_model_run_name = model_run_folder_name + f'{model_run_name}_{col}_{model_target}.pkl'
            joblib.dump(model, path_to_model_run_name)

        cv_results_df = pd.DataFrame(search_obj.cv_results_)
        cv_results_df['orientation_data'] = col
        cv_results_list.append(cv_results_df)

    master_cv_results_df = pd.concat(cv_results_list)
    master_cv_results_df['model_target'] = model_target
    master_cv_results_df['train_size'] = train_size
    master_cv_results_df['is_random_search'] = random_search
    path_to_cv_results = model_run_folder_name + f'{model_run_name}_{model_target}_results.csv'
    master_cv_results_df.to_csv(path_to_cv_results, index=False)

Fitting 3 folds for each of 3 candidates, totalling 9 fits
